In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'Street Name',
                    'Address Type', 'City', 'Community Board', 'Borough',
                    'Open Data Channel Type', 'Park Borough', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month',
                    'is_weekend', 'open_requests_same_day_zip']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'Street Name',
                    'Address Type', 'City', 'Community Board', 'Borough',
                    'Open Data Channel Type', 'Park Borough',
                    'is_weekend']
numeric_cols = ['Latitude','Longitude','hour_of_day','day_of_week', 'month','open_requests_same_day_zip']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

lr = LogisticRegression(
    solver='saga',      # MUCH faster for big sparse data
    max_iter=200,       # enough for saga
    tol=0.005,
    class_weight='balanced',
    n_jobs=-1,          # use all CPU cores
    verbose=3           # show progress
)

# 4. Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', lr)
])

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
print(X.shape)
print(y.shape)
df.dtypes